# INT8 PTQ 양자화 재검증

**목표**: `best_1dcnn.pth` (FP32 SkinCNN) → INT8 Post-Training Static Quantization → LOO-CV 재평가

**방법**
- Backend: `qnnpack` (ARM aarch64 지원)
- 융합: Conv1d + BatchNorm1d + ReLU → ConvBnReLU1d
- 보정(Calibration): 각 LOO fold의 학습 데이터 전체
- 평가: FP32 vs INT8 MAE, 추론 속도, 모델 크기 비교

In [ ]:
import warnings, re, json, time, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── 경고 억제 (deprecated eager-mode quantization API)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import torch.quantization  # noqa: E402

# ── 한글 폰트
_ko = [f.fname for f in fm.fontManager.ttflist if 'Noto Sans CJK' in f.name]
if _ko:
    fm.fontManager.addfont(_ko[0])
    _kn = fm.FontProperties(fname=_ko[0]).get_name()
    plt.rcParams.update({'font.family': 'sans-serif',
                         'font.sans-serif': [_kn, 'DejaVu Sans']})
plt.rcParams['axes.unicode_minus'] = False

# ── 경로 / 상수
ROOT       = Path('.').resolve()
USDATA     = ROOT / 'usdata' / 'data'
RESULT_DIR = ROOT / 'results'

torch.backends.quantized.engine = 'qnnpack'
DEVICE = torch.device('cpu')          # PTQ는 CPU에서 수행

TRIM_START, TRIM_COUNT = 1200, 1250
SOUND_SPEED = 1540.0
CALIB_BATCH = 64                      # 보정 배치 크기

FILENAME_RE = re.compile(
    r'^(?P<name>.+)_(?P<date>\d{8})_(?P<time>\d{6})_'
    r'\((?P<pos_num>\d+)\)(?P<position>.+?)_(?P<gender>[MF])\.csv$'
)

print(f'PyTorch : {torch.__version__}')
print(f'Backend : {torch.backends.quantized.engine}')
print(f'USDATA  : {USDATA}')

## 1. 데이터 로드

In [ ]:
def load_adc(path):
    adc = []
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                v = int(line)
                if 0 <= v <= 255: adc.append(v)
            except ValueError: pass
    adc = np.array(adc, dtype=np.float32)
    if len(adc) > TRIM_COUNT:
        end = TRIM_START + TRIM_COUNT
        adc = adc[TRIM_START:end] if len(adc) >= end else adc[TRIM_START:]
    if len(adc) < TRIM_COUNT:
        adc = np.pad(adc, (0, TRIM_COUNT - len(adc)))
    return adc[:TRIM_COUNT]


def load_label(csv_path):
    json_path = csv_path.with_name(csv_path.stem + '_positions.json')
    if not json_path.exists(): return None
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)
    dermis_mm = fascia_mm = None
    for pos in data.get('positions', []):
        name = pos.get('position_name', '')
        mm   = pos.get('thickness_mm')
        if name in ('피하지방시작', 'Dermis') and mm is not None:
            dermis_mm = float(mm)
        elif name == 'Fascia' and mm is not None:
            fascia_mm = float(mm)
    if dermis_mm is None or fascia_mm is None: return None
    return dermis_mm, fascia_mm


def build_dataset():
    rows = []
    for csv_path in sorted(USDATA.rglob('*.csv')):
        if '_positions' in csv_path.name: continue
        m = FILENAME_RE.match(csv_path.name)
        if not m: continue
        label = load_label(csv_path)
        if label is None: continue
        adc = load_adc(csv_path)
        rows.append({
            'patient': m.group('name'),
            'pos_num': int(m.group('pos_num')),
            'adc':     adc,
            'dermis':  label[0],
            'fascia':  label[1],
        })
    return rows


class UltrasoundDataset(Dataset):
    def __init__(self, rows):
        x = np.stack([(r['adc'] - 128.0) / 128.0 for r in rows])
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor([[r['dermis'], r['fascia']] for r in rows],
                              dtype=torch.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]


ALL_ROWS = build_dataset()
patients = sorted({r['patient'] for r in ALL_ROWS})
print(f'총 샘플: {len(ALL_ROWS)}개  /  환자: {len(patients)}명')

## 2. 모델 정의 — SkinCNN (FP32) + SkinCNNQ (양자화 가능)

In [ ]:
class SkinCNN(nn.Module):
    """원본 FP32 모델 (학습용)"""
    def __init__(self, n_filters=16, dropout=0.3):
        super().__init__()
        f = n_filters
        self.encoder = nn.Sequential(
            nn.Conv1d(1,   f,   15, stride=2, padding=7),  nn.BatchNorm1d(f),   nn.ReLU(),
            nn.Conv1d(f,   f*2, 7,  stride=2, padding=3),  nn.BatchNorm1d(f*2), nn.ReLU(),
            nn.Conv1d(f*2, f*4, 5,  stride=2, padding=2),  nn.BatchNorm1d(f*4), nn.ReLU(),
            nn.Conv1d(f*4, f*4, 5,  stride=2, padding=2),  nn.BatchNorm1d(f*4), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(f*4, f*2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(f*2, 2),
        )
    def forward(self, x): return self.head(self.encoder(x))


class SkinCNNQ(nn.Module):
    """PTQ 정적 양자화용 — Conv+BN+ReLU 명시적 분리 + QuantStub/DeQuantStub"""

    # encoder.X → named layer 매핑
    _WEIGHT_MAP = {
        'encoder.0': 'conv1', 'encoder.1': 'bn1',
        'encoder.3': 'conv2', 'encoder.4': 'bn2',
        'encoder.6': 'conv3', 'encoder.7': 'bn3',
        'encoder.9': 'conv4', 'encoder.10': 'bn4',
        'head.1': 'fc1', 'head.4': 'fc2',
    }

    def __init__(self, n_filters=16, dropout=0.3):
        super().__init__()
        f = n_filters
        self.quant  = torch.quantization.QuantStub()

        self.conv1 = nn.Conv1d(1,   f,   15, stride=2, padding=7)
        self.bn1   = nn.BatchNorm1d(f)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(f,   f*2, 7,  stride=2, padding=3)
        self.bn2   = nn.BatchNorm1d(f*2)
        self.relu2 = nn.ReLU()

        self.conv3 = nn.Conv1d(f*2, f*4, 5,  stride=2, padding=2)
        self.bn3   = nn.BatchNorm1d(f*4)
        self.relu3 = nn.ReLU()

        self.conv4 = nn.Conv1d(f*4, f*4, 5,  stride=2, padding=2)
        self.bn4   = nn.BatchNorm1d(f*4)
        self.relu4 = nn.ReLU()

        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(f*4, f*2)
        self.relu5   = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(f*2, 2)

        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.relu4(self.bn4(self.conv4(x)))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu5(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.dequant(x)

    def fuse_model(self):
        """Conv+BN+ReLU / Linear+ReLU 융합 (양자화 효율 향상)"""
        torch.quantization.fuse_modules(self, [
            ['conv1', 'bn1', 'relu1'],
            ['conv2', 'bn2', 'relu2'],
            ['conv3', 'bn3', 'relu3'],
            ['conv4', 'bn4', 'relu4'],
            ['fc1', 'relu5'],
        ], inplace=True)

    @classmethod
    def from_skincnn(cls, fp32_model, n_filters=16, dropout=0.3):
        """SkinCNN(Sequential) 가중치 → SkinCNNQ(named layers) 복사"""
        q = cls(n_filters=n_filters, dropout=dropout)
        sd = fp32_model.state_dict()
        new_sd = {}
        for old_prefix, new_prefix in cls._WEIGHT_MAP.items():
            for k, v in sd.items():
                if k.startswith(old_prefix + '.'):
                    new_key = new_prefix + k[len(old_prefix):]
                    new_sd[new_key] = v
        missing, unexpected = q.load_state_dict(new_sd, strict=False)
        if missing:
            print(f'  [경고] missing keys: {missing}')
        return q


print(f'SkinCNN   파라미터: {sum(p.numel() for p in SkinCNN(16).parameters()):,}개')
print(f'SkinCNNQ  파라미터: {sum(p.numel() for p in SkinCNNQ(16).parameters()):,}개')

## 3. PTQ 유틸리티

In [ ]:
def calibrate(model_q, calib_rows, batch=CALIB_BATCH):
    """보정 데이터로 activation 범위 수집"""
    loader = DataLoader(UltrasoundDataset(calib_rows), batch_size=batch, shuffle=False)
    model_q.eval()
    with torch.no_grad():
        for x, _ in loader:
            model_q(x)


def apply_ptq(fp32_model, calib_rows, n_filters=16, dropout=0.3):
    """FP32 SkinCNN → INT8 SkinCNNQ
    1) 가중치 복사  2) 융합  3) 보정  4) 변환
    Returns: quantized model (eval mode)
    """
    model_q = SkinCNNQ.from_skincnn(fp32_model, n_filters=n_filters, dropout=dropout)
    model_q.eval()
    model_q.fuse_model()
    model_q.qconfig = torch.quantization.get_default_qconfig('qnnpack')
    torch.quantization.prepare(model_q, inplace=True)
    calibrate(model_q, calib_rows)
    torch.quantization.convert(model_q, inplace=True)
    return model_q


@torch.no_grad()
def evaluate_mae(model, rows):
    """MAE (진피, 근막) 반환. preds, trues도 함께 반환."""
    model.eval()
    loader = DataLoader(UltrasoundDataset(rows), batch_size=64)
    preds, trues = [], []
    for x, y in loader:
        preds.append(model(x).numpy())
        trues.append(y.numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    mae_d = float(np.mean(np.abs(preds[:, 0] - trues[:, 0])))
    mae_f = float(np.mean(np.abs(preds[:, 1] - trues[:, 1])))
    return mae_d, mae_f, preds, trues


def model_size_kb(model):
    """모델 파라미터 메모리 (KB)"""
    total = 0
    for p in model.parameters():
        # INT8은 element_size()=1, FP32=4
        total += p.nelement() * p.element_size()
    # buffers (BN running stats 등)
    for b in model.buffers():
        total += b.nelement() * b.element_size()
    return total / 1024


def measure_latency(model, n=200, seq_len=1250):
    """단일 샘플 평균 추론 시간 (ms)"""
    model.eval()
    x = torch.randn(1, 1, seq_len)
    with torch.no_grad():
        for _ in range(20):  # warmup
            model(x)
        t0 = time.perf_counter()
        for _ in range(n):
            model(x)
        return (time.perf_counter() - t0) / n * 1000


print('유틸리티 정의 완료')

## 4. 저장된 모델로 빠른 PTQ 검증

`best_1dcnn.pth` (전체 데이터 학습) → PTQ → 마지막 환자 검증  
FP32 vs INT8 MAE + 속도 + 크기 비교

In [ ]:
ckpt = torch.load(RESULT_DIR / 'best_1dcnn.pth', map_location='cpu')
hp   = ckpt['hyperparams']

fp32_best = SkinCNN(n_filters=hp['n_filters'], dropout=hp['dropout'])
fp32_best.load_state_dict(ckpt['model_state'])
fp32_best.eval()

# 빠른 검증: 마지막 환자 held-out
val_pat   = patients[-1]
calib_rows = [r for r in ALL_ROWS if r['patient'] != val_pat]
val_rows   = [r for r in ALL_ROWS if r['patient'] == val_pat]

print(f'보정 데이터: {len(calib_rows)}개  /  검증(held-out): {len(val_rows)}개 ({val_pat})')

# FP32 평가
mae_d_fp, mae_f_fp, _, _ = evaluate_mae(fp32_best, val_rows)
lat_fp = measure_latency(fp32_best)
sz_fp  = model_size_kb(fp32_best)

# PTQ 적용
print('PTQ 적용 중...')
int8_best = apply_ptq(fp32_best, calib_rows,
                      n_filters=hp['n_filters'], dropout=hp['dropout'])

mae_d_q, mae_f_q, _, _ = evaluate_mae(int8_best, val_rows)
lat_q  = measure_latency(int8_best)
sz_q   = model_size_kb(int8_best)

print()
print(f"{'':20s}  {'진피 MAE':>10s}  {'근막 MAE':>10s}  {'속도(ms)':>10s}  {'크기(KB)':>10s}")
print('-' * 68)
print(f"{'FP32':20s}  {mae_d_fp:>10.4f}  {mae_f_fp:>10.4f}  {lat_fp:>10.3f}  {sz_fp:>10.1f}")
print(f"{'INT8 PTQ':20s}  {mae_d_q:>10.4f}  {mae_f_q:>10.4f}  {lat_q:>10.3f}  {sz_q:>10.1f}")
print()
print(f'진피 MAE 변화: {mae_d_q - mae_d_fp:+.4f} mm  ({(mae_d_q/mae_d_fp-1)*100:+.1f}%)')
print(f'근막 MAE 변화: {mae_f_q - mae_f_fp:+.4f} mm  ({(mae_f_q/mae_f_fp-1)*100:+.1f}%)')
print(f'속도 향상: {lat_fp/lat_q:.2f}x')
print(f'크기 감소: {sz_fp/sz_q:.2f}x')

## 5. LOO-CV 재검증 — FP32 vs INT8 전 환자 비교

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0
    for x, y in loader:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(x)
    return total / len(loader.dataset)


def train_fp32(train_rows, val_rows, hp, title=''):
    """SkinCNN FP32 학습 → best val 모델 반환"""
    loader_tr = DataLoader(UltrasoundDataset(train_rows),
                           batch_size=hp['batch'], shuffle=True,
                           drop_last=len(train_rows) > hp['batch'])
    loader_val = DataLoader(UltrasoundDataset(val_rows), batch_size=64)

    model = SkinCNN(hp['n_filters'], hp['dropout'])
    opt   = torch.optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=hp['epochs'])
    crit  = nn.L1Loss()

    best_val, best_state, no_imp = float('inf'), None, 0
    for ep in range(1, hp['epochs'] + 1):
        train_epoch(model, loader_tr, opt, crit)
        val_loss, _, _, _, _ = _eval(model, loader_val, crit)
        sched.step()
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= hp['patience']:
            break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def _eval(model, loader, criterion):
    model.eval()
    total, preds, trues = 0, [], []
    for x, y in loader:
        pred = model(x)
        total += criterion(pred, y).item() * len(x)
        preds.append(pred.numpy())
        trues.append(y.numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    mae_d = float(np.mean(np.abs(preds[:,0]-trues[:,0])))
    mae_f = float(np.mean(np.abs(preds[:,1]-trues[:,1])))
    return total/len(loader.dataset), mae_d, mae_f, preds, trues


print('학습/평가 함수 정의 완료')

In [ ]:
# ──────────────────────────────────────────────────────────
#  LOO-CV: FP32 학습 → INT8 PTQ → 양쪽 MAE 기록
# ──────────────────────────────────────────────────────────
LOO_HP = dict(
    lr=1e-3, batch=32, n_filters=16, dropout=0.3,
    epochs=80, patience=20
)

records = []
print(f'LOO-CV 시작: {len(patients)}명  (epochs={LOO_HP["epochs"]})')
print('-' * 75)
print(f'{"":20s}  {"n":>4s}  {"FP32 진피":>9s}  {"FP32 근막":>9s}  '
      f'{"INT8 진피":>9s}  {"INT8 근막":>9s}  {"진피Δ":>7s}  {"근막Δ":>7s}')
print('-' * 75)

for i, pat in enumerate(patients):
    tr  = [r for r in ALL_ROWS if r['patient'] != pat]
    val = [r for r in ALL_ROWS if r['patient'] == pat]

    # 1) FP32 학습
    fp32 = train_fp32(tr, val, LOO_HP)
    mae_d_fp, mae_f_fp, preds_fp, trues = evaluate_mae(fp32, val)

    # 2) PTQ: 보정은 학습 데이터(tr) 전체 사용
    int8 = apply_ptq(fp32, tr, n_filters=LOO_HP['n_filters'],
                     dropout=LOO_HP['dropout'])
    mae_d_q, mae_f_q, preds_q, _ = evaluate_mae(int8, val)

    delta_d = mae_d_q - mae_d_fp
    delta_f = mae_f_q - mae_f_fp

    print(f'{f"[{i+1:2d}/{len(patients)}] {pat}":<20s}  '
          f'{len(val):>4d}  '
          f'{mae_d_fp:>9.3f}  {mae_f_fp:>9.3f}  '
          f'{mae_d_q:>9.3f}  {mae_f_q:>9.3f}  '
          f'{delta_d:>+7.3f}  {delta_f:>+7.3f}')

    records.append({
        'patient':    pat,
        'n':          len(val),
        'fp32_d':     mae_d_fp, 'fp32_f': mae_f_fp,
        'int8_d':     mae_d_q,  'int8_f': mae_f_q,
        'delta_d':    delta_d,  'delta_f': delta_f,
        'preds_fp':   preds_fp, 'preds_q': preds_q, 'trues': trues,
    })

CV = pd.DataFrame(records)
print('-' * 75)
print(f'{"평균":20s}  '
      f'{"":>4s}  '
      f'{CV.fp32_d.mean():>9.3f}  {CV.fp32_f.mean():>9.3f}  '
      f'{CV.int8_d.mean():>9.3f}  {CV.int8_f.mean():>9.3f}  '
      f'{CV.delta_d.mean():>+7.3f}  {CV.delta_f.mean():>+7.3f}')
print(f'{"표준편차":20s}  '
      f'{"":>4s}  '
      f'{CV.fp32_d.std():>9.3f}  {CV.fp32_f.std():>9.3f}  '
      f'{CV.int8_d.std():>9.3f}  {CV.int8_f.std():>9.3f}')

## 6. 결과 시각화

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.suptitle('INT8 PTQ 양자화 재검증 — LOO-CV 결과', fontsize=14)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

TARGET = 0.30

# ① 환자별 MAE 막대 (진피)
ax = fig.add_subplot(gs[0, 0])
x = np.arange(len(CV))
ax.bar(x - 0.2, CV.fp32_d, 0.35, label='FP32', color='#4c72b0', alpha=0.85)
ax.bar(x + 0.2, CV.int8_d, 0.35, label='INT8', color='#c44e52', alpha=0.85)
ax.axhline(TARGET, color='red', linestyle='--', linewidth=1, label='목표 0.30mm')
ax.set_xticks(x)
ax.set_xticklabels(CV.patient, rotation=45, ha='right', fontsize=6)
ax.set_title('환자별 진피 MAE')
ax.set_ylabel('mm')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.35)

# ② 환자별 MAE 막대 (근막)
ax = fig.add_subplot(gs[0, 1])
ax.bar(x - 0.2, CV.fp32_f, 0.35, label='FP32', color='#4c72b0', alpha=0.85)
ax.bar(x + 0.2, CV.int8_f, 0.35, label='INT8', color='#c44e52', alpha=0.85)
ax.axhline(TARGET, color='red', linestyle='--', linewidth=1, label='목표 0.30mm')
ax.set_xticks(x)
ax.set_xticklabels(CV.patient, rotation=45, ha='right', fontsize=6)
ax.set_title('환자별 근막 MAE')
ax.set_ylabel('mm')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.35)

# ③ Δ MAE 분포 (INT8 - FP32)
ax = fig.add_subplot(gs[0, 2])
ax.boxplot([CV.delta_d, CV.delta_f],
           tick_labels=['진피 Δ', '근막 Δ'],
           patch_artist=True,
           boxprops=dict(facecolor='#dd8452', alpha=0.7))
ax.axhline(0, color='gray', linewidth=1)
ax.set_title('INT8 - FP32 MAE 변화량')
ax.set_ylabel('Δ mm (양수=성능저하)')
ax.grid(axis='y', alpha=0.35)

# ④ 예측 vs 실제 (FP32 진피)
all_preds_fp = np.concatenate([r['preds_fp'] for r in records])
all_preds_q  = np.concatenate([r['preds_q']  for r in records])
all_trues    = np.concatenate([r['trues']     for r in records])

for col_idx, (preds, label_str, color) in enumerate([
        (all_preds_fp[:, 0], 'FP32 진피', '#4c72b0'),
        (all_preds_q[:, 0],  'INT8 진피', '#c44e52'),
    ]):
    ax = fig.add_subplot(gs[1, col_idx])
    trues_d = all_trues[:, 0]
    ax.scatter(trues_d, preds, alpha=0.3, s=12, color=color)
    lim = [trues_d.min()-0.05, trues_d.max()+0.05]
    ax.plot(lim, lim, 'r--', linewidth=1)
    mae = float(np.mean(np.abs(preds - trues_d)))
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('실제 진피 (mm)')
    ax.set_ylabel(f'예측 진피 (mm)')
    ax.set_title(f'{label_str}  MAE={mae:.3f}mm')
    ax.grid(alpha=0.3)

# ⑤ 요약 표
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
table_data = [
    ['항목', 'FP32', 'INT8', '변화'],
    ['진피 MAE (평균)',
     f'{CV.fp32_d.mean():.3f}mm',
     f'{CV.int8_d.mean():.3f}mm',
     f'{CV.delta_d.mean():+.3f}mm'],
    ['진피 MAE (std)',
     f'±{CV.fp32_d.std():.3f}',
     f'±{CV.int8_d.std():.3f}', ''],
    ['근막 MAE (평균)',
     f'{CV.fp32_f.mean():.3f}mm',
     f'{CV.int8_f.mean():.3f}mm',
     f'{CV.delta_f.mean():+.3f}mm'],
    ['근막 MAE (std)',
     f'±{CV.fp32_f.std():.3f}',
     f'±{CV.int8_f.std():.3f}', ''],
    ['목표 달성 (진피<0.3)',
     f"{(CV.fp32_d < TARGET).sum()}/{len(CV)}",
     f"{(CV.int8_d < TARGET).sum()}/{len(CV)}", ''],
    ['목표 달성 (근막<0.3)',
     f"{(CV.fp32_f < TARGET).sum()}/{len(CV)}",
     f"{(CV.int8_f < TARGET).sum()}/{len(CV)}", ''],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc='center', loc='center',
               colWidths=[0.32, 0.22, 0.22, 0.24])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.6)
ax.set_title('LOO-CV 요약', fontsize=10, pad=10)

out = RESULT_DIR / 'quantize_eval.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'저장: {out}')
plt.show()

## 7. 최종 INT8 모델 저장 (전체 데이터 재학습 → PTQ)

In [ ]:
print('최종 FP32 모델 학습 중 (전체 데이터)...')
final_hp = {**LOO_HP, 'epochs': 200, 'patience': 30}
# 검증은 마지막 환자 (조기종료 기준만, 최종 모델은 전체 사용)
val_last = [r for r in ALL_ROWS if r['patient'] == patients[-1]]
tr_all   = ALL_ROWS  # 전체 데이터로 학습

fp32_final = train_fp32(ALL_ROWS, val_last, final_hp)

print('PTQ 적용 (보정: 전체 데이터)...')
int8_final = apply_ptq(fp32_final, ALL_ROWS,
                       n_filters=final_hp['n_filters'],
                       dropout=final_hp['dropout'])

# 저장
torch.save(int8_final, RESULT_DIR / 'best_1dcnn_int8.pth')
print(f'INT8 모델 저장: {RESULT_DIR}/best_1dcnn_int8.pth')

# 속도 / 크기 측정
lat_fp = measure_latency(fp32_final)
lat_q  = measure_latency(int8_final)
sz_fp  = model_size_kb(fp32_final)
sz_q   = model_size_kb(int8_final)

print()
print('=== 최종 비교 ===')
print(f'{"":20s}  {"진피 MAE":>10s}  {"근막 MAE":>10s}  {"속도(ms/샘플)":>14s}  {"크기(KB)":>10s}')
print('-' * 72)
print(f'{"FP32 (LOO-CV 평균)":20s}  '
      f'{CV.fp32_d.mean():>10.3f}  {CV.fp32_f.mean():>10.3f}  '
      f'{lat_fp:>14.3f}  {sz_fp:>10.1f}')
print(f'{"INT8 (LOO-CV 평균)":20s}  '
      f'{CV.int8_d.mean():>10.3f}  {CV.int8_f.mean():>10.3f}  '
      f'{lat_q:>14.3f}  {sz_q:>10.1f}')
print()
print(f'속도 향상: {lat_fp/lat_q:.2f}x  │  크기 감소: {sz_fp/sz_q:.2f}x')
ok_d = (CV.int8_d < 0.3).all()
ok_f = (CV.int8_f < 0.3).all()
print(f'목표 <0.30mm 달성: 진피={"✅" if ok_d else "❌"}  근막={"✅" if ok_f else "❌"}')